In [56]:
import geopandas as gpd
import pandas as pd
import numpy as np
from bld_src import config

In [ ]:
## data types: object (string), float, int

# indexing: [row,column]

In [2]:
plots = gpd.read_file(config.DATA_RAW / 'Teatown Sites.shp' / 'Points.shp')

In [ ]:
plots = plots[['Name','geometry']]

# separate out trees and sites
trees = plots.loc[plots['Name'].str.contains('Tree')].copy()
sites = plots.loc[~plots['Name'].str.contains('Tree')].copy()
sites.to_file(config.DATA_PROCESSED / 'teatown_sites_points.gpkg')

# clean name column (list comprehension)
trees['Name'] = [n.split(' ')[1] for n in trees['Name']]
trees['Name'] = trees['Name'].astype(int)

In [ ]:
eval = pd.read_excel(config.DATA_RAW / 'BLD_Evaluation_2026_Data.xlsx')
# merge points with health data
trees_eval = trees.merge(eval,left_on='Name',right_on='Tree #')
trees_eval = trees_eval.drop(columns={'Tree #'})
# rename columns
trees_eval.columns = ['Name','geometry','dbh','species','crown_class','leaf_density','dieback_overall','leaf_discolor','dead_branch_main','dead_branch_fine','no_symptom','banded','curled','bbd_present']
#trees_eval.to_file(config.DATA_PROCESSED / 'teatown_trees_eval.gpkg')

## basal area column
trees_eval['basal_area'] = config.BA_CONVERSION * trees_eval['dbh']**2

# write to file
trees_eval.to_file(config.DATA_PROCESSED / 'teatown_trees_eval.gpkg')

In [ ]:
## convert tree-level health metrics to plot-level 

## code nonbeech as 0 for bld metrics
bld_metrics = ['no_symptom','banded','curled']
trees_eval= trees_eval.fillna({m:0.0 for m in bld_metrics})

is_beech = trees_eval['species'] == 'Beech'

# weight metrics by basal area
for m in bld_metrics:
    trees_eval[f'{m}_weighted'] = np.where(is_beech,trees_eval[m]*trees_eval['basal_area'],0.0)
    trees_eval['nonbeech_dieback_weighted'] = np.where(~is_beech,trees_eval['dieback_overall']*trees_eval['basal_area'],0.0)

# make basal area column for beech only; nonbeech gets set to 0
trees_eval['beech_ba'] = np.where(is_beech,trees_eval['basal_area'],0.0)

In [ ]:
### this cell groups trees by plot and calculates plot-level health metrics
### will only run if plot column exists!

plot_col = ' '

# group trees by plot
grouped = trees_eval.groupby(plot_col)
out = pd.DataFrame({'total_ba':grouped['basal_area'].sum(),  ## total basal area per plot
                'beech_ba' : grouped['beech_ba'].sum(), # total beech basal area per plot
                    'nonbeech_ba': grouped['nonbeech_ba'].sum(),   # nonbeech basal area per plot
                    'nonbeech_dieback': grouped['nonbeech_dieback_weighted'].sum(),  ## plot level health for non-beech
                    'n_beech': grouped['species'].apply(lambda x: (x=='Beech').sum()), # number of beech per plot
                    'n_nonbeech': grouped['species'].apply(lambda x: (x!='Beech').sum())})  # number of nonbeech per plot
for m in bld_metrics:
    out[m] = grouped[f'{m}_weighted'].sum()   ## sum of weight beech health metrics per plot (no symptom, banded, curled)

# beech abundance
out['beech_rel_ba'] = out['beech_ba'] / out['total_ba']  

# weighted summed per-tree metrics divided by plot basal area
for m in bld_metrics:
    out[f'{m}_burden'] = out[m] / out['total_ba'] ## plot wide disease burden metric
    out[f'{m}_mean_severity'] = out[m] / out['beech_ba'] ## beech only severity metric

out['nonbeech_dieback_burden'] = out['nonbeech_dieback'] / out['total_ba']

out = out.reset_index()


